# EDA de Referências — AI Governance in Brazilian Universities

**Projeto:** `ai-governance-brazilian-universities`  
**Insumo:** `data/processed/referencias_kwic.csv` (revisado manualmente)  

Este notebook analisa as referências citadas nos 20 documentos do corpus em três eixos:

1. **Top referências mais citadas** — quais marcos normativos e autores são mais mobilizados
2. **Referências únicas por instituição** — quais documentos têm base referencial mais ampla
3. **Referências compartilhadas** — pontos de convergência entre instituições

---

In [ ]:
# ── Dependências ──────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT       = Path("..")
KWIC_PATH  = ROOT / "data" / "processed" / "referencias_kwic.csv"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Estilo global ─────────────────────────────────────────────────────────────
CORES_TIPO = {
    "Oficial nacional":       "#1A8C6A",
    "Oficial internacional":  "#C0392B",
    "Acadêmico nacional":     "#E67E22",
    "Acadêmico internacional":"#8E44AD",
}
GREY = "#AAAAAA"

sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams.update({
    "figure.dpi":        150,
    "figure.facecolor":  "white",
    "axes.facecolor":    "white",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.spines.left":  False,
    "axes.grid":         True,
    "grid.color":        "#E5E5E5",
    "grid.linewidth":    0.6,
})

print("Ambiente pronto.")

---
## Carga e filtragem

In [ ]:
df = pd.read_csv(KWIC_PATH, encoding="utf-8-sig")
df.columns = df.columns.str.strip()

for col in ["instituicao", "arquivo_pdf", "referencia_limpa", "valido", "nota"]:
    df[col] = df[col].fillna("").astype(str).str.strip()

# Apenas referências válidas com referencia_limpa preenchida
refs = df[
    (df["valido"].str.lower() == "sim") &
    (df["referencia_limpa"] != "")
].copy()

refs["tipo"] = refs["nota"].replace("", "Não classificado")

print(f"Referências válidas: {len(refs)}")
print()
print("Por tipo:")
print(refs["tipo"].value_counts().to_string())

---
## Parte 1 — Top referências mais citadas

In [ ]:
TOP_N = 20

top_refs = (
    refs.groupby(["referencia_limpa", "tipo"])
    .size()
    .reset_index(name="n_citacoes")
    .sort_values("n_citacoes", ascending=False)
    .head(TOP_N)
    .reset_index(drop=True)
)

# Trunca rótulos longos para legibilidade
top_refs["label"] = top_refs["referencia_limpa"].str[:60]

cores = [CORES_TIPO.get(t, GREY) for t in top_refs["tipo"]]

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(top_refs["label"][::-1], top_refs["n_citacoes"][::-1],
               color=cores[::-1], height=0.65)

for bar, val in zip(bars, top_refs["n_citacoes"][::-1]):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
            str(val), va="center", fontsize=9, fontweight="bold")

# Legenda
handles = [mpatches.Patch(color=cor, label=tipo)
           for tipo, cor in CORES_TIPO.items()]
ax.legend(handles=handles, fontsize=8, loc="lower right", framealpha=0.9)

ax.set_xlabel("Número de citações", labelpad=8)
ax.set_title(f"Top {TOP_N} referências mais citadas",
             fontsize=13, fontweight="bold", pad=12)
ax.set_xlim(0, top_refs["n_citacoes"].max() + 2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "refs_01_top_citadas.png", bbox_inches="tight")
plt.show()

---
## Parte 2 — Distribuição por tipo de referência

In [ ]:
# Total de citações por tipo
por_tipo = (
    refs.groupby("tipo")
    .size()
    .reset_index(name="n")
    .sort_values("n", ascending=False)
)
cores_tipo = [CORES_TIPO.get(t, GREY) for t in por_tipo["tipo"]]

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.barh(por_tipo["tipo"][::-1], por_tipo["n"][::-1],
               color=cores_tipo[::-1], height=0.55)
for bar, val in zip(bars, por_tipo["n"][::-1]):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
            str(val), va="center", fontsize=10, fontweight="bold")

ax.set_xlabel("Citações", labelpad=8)
ax.set_title("Citações por tipo de referência",
             fontsize=13, fontweight="bold", pad=12)
ax.set_xlim(0, por_tipo["n"].max() + 5)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "refs_02_por_tipo.png", bbox_inches="tight")
plt.show()

---
## Parte 3 — Referências únicas por instituição

In [ ]:
refs_unicas = (
    refs.groupby("instituicao")["referencia_limpa"]
    .nunique()
    .reset_index(name="refs_unicas")
    .sort_values("refs_unicas", ascending=True)
)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(refs_unicas["instituicao"], refs_unicas["refs_unicas"],
               color="#2D5FA8", height=0.6)
for bar, val in zip(bars, refs_unicas["refs_unicas"]):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
            str(val), va="center", fontsize=9, fontweight="bold")

ax.set_xlabel("Referências únicas", labelpad=8)
ax.set_title("Referências únicas por instituição",
             fontsize=13, fontweight="bold", pad=12)
ax.set_xlim(0, refs_unicas["refs_unicas"].max() + 3)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "refs_03_unicas_por_inst.png", bbox_inches="tight")
plt.show()

---
## Parte 4 — Composição do repertório por instituição

In [ ]:
# Stacked bar: cada instituição mostra quantas citações de cada tipo
composicao = (
    refs.groupby(["instituicao", "tipo"])
    .size()
    .unstack(fill_value=0)
)

# Ordena por total de citações
composicao["total"] = composicao.sum(axis=1)
composicao = composicao.sort_values("total", ascending=True).drop(columns="total")

# Cores alinhadas às colunas
cols_plot = [c for c in composicao.columns if c in CORES_TIPO]
cores_plot = [CORES_TIPO[c] for c in cols_plot]

fig, ax = plt.subplots(figsize=(10, 6))
composicao[cols_plot].plot(
    kind="barh", stacked=True, ax=ax,
    color=cores_plot, width=0.65, edgecolor="white"
)

ax.set_xlabel("Número de citações", labelpad=8)
ax.set_title("Composição do repertório de referências por instituição",
             fontsize=13, fontweight="bold", pad=12)
ax.legend(title="Tipo", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "refs_04_composicao_por_inst.png", bbox_inches="tight")
plt.show()

---
## Parte 5 — Referências compartilhadas por mais de uma instituição

In [ ]:
compartilhadas = (
    refs.groupby(["referencia_limpa", "tipo"])["instituicao"]
    .nunique()
    .reset_index(name="n_instituicoes")
)
compartilhadas = (
    compartilhadas[compartilhadas["n_instituicoes"] > 1]
    .sort_values("n_instituicoes", ascending=True)
    .reset_index(drop=True)
)

compartilhadas["label"] = compartilhadas["referencia_limpa"].str[:55]
cores = [CORES_TIPO.get(t, GREY) for t in compartilhadas["tipo"]]

fig, ax = plt.subplots(figsize=(10, max(4, len(compartilhadas) * 0.45)))
bars = ax.barh(compartilhadas["label"], compartilhadas["n_instituicoes"],
               color=cores, height=0.65)
for bar, val in zip(bars, compartilhadas["n_instituicoes"]):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height() / 2,
            str(val), va="center", fontsize=9, fontweight="bold")

handles = [mpatches.Patch(color=cor, label=tipo)
           for tipo, cor in CORES_TIPO.items()
           if tipo in compartilhadas["tipo"].values]
ax.legend(handles=handles, fontsize=8, loc="lower right", framealpha=0.9)

ax.set_xlabel("Número de instituições que citam", labelpad=8)
ax.set_title("Referências compartilhadas por mais de uma instituição",
             fontsize=13, fontweight="bold", pad=12)
ax.set_xlim(0, compartilhadas["n_instituicoes"].max() + 1.5)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "refs_05_compartilhadas.png", bbox_inches="tight")
plt.show()

print(f"\nTotal de referências compartilhadas: {len(compartilhadas)}")